# Code 4: Threshold Selection Analysis
Runs CE + LLM on ALL samples, then simulates different thresholds.
This justifies the choice of threshold used in the hybrid pipeline.

In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder
from google import genai
from sklearn.metrics import accuracy_score

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ============================================================
# Configuration
# ============================================================
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["GOOGLE_API_KEY"] = "Your API Key Here"  # Set your API key in environment variable

CE_MODEL_NAME = "cross-encoder/nli-distilroberta-base"
LLM_MODEL = "gemini-2.0-flash"
DATA_FILE = "multinli_1.0_dev_matched.jsonl"
TEST_SIZE = 50          
MAX_RETRIES = 6
REQUEST_DELAY = 4
VALID_LABELS = ["contradiction", "entailment", "neutral"]

# ============================================================
# Data Loading
# ============================================================
def load_data(path, n):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if len(samples) >= n:
                break
            try:
                obj = json.loads(line.strip())
                if obj.get("gold_label") in VALID_LABELS:
                    samples.append(obj)
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(samples)} samples.")
    return samples

# ============================================================
# Cross-Encoder
# ============================================================
def run_cross_encoder(samples):
    print("Loading Cross-Encoder...")
    ce = CrossEncoder(CE_MODEL_NAME, device="cpu")
    pairs = [(s["sentence1"], s["sentence2"]) for s in samples]
    scores = ce.predict(pairs, show_progress_bar=True)

    preds, confs = [], []
    for score in scores:
        probs = np.exp(score - np.max(score))
        probs = probs / probs.sum()
        preds.append(VALID_LABELS[np.argmax(probs)])
        confs.append(np.max(probs))
    return np.array(preds), np.array(confs)

# ============================================================
# LLM (Zero-Shot on ALL samples — needed for fair sweep)
# ============================================================
def build_zero_shot(premise, hypothesis):
    return (
        "You are an NLI classifier. Determine if the hypothesis is an "
        "'entailment', 'contradiction', or 'neutral' to the premise.\n"
        "- entailment: the hypothesis MUST be true given the premise.\n"
        "- contradiction: the hypothesis CANNOT be true given the premise.\n"
        "- neutral: the hypothesis MIGHT be true, but the premise doesn't "
        "give enough info to be sure.\n\n"
        "Respond with EXACTLY ONE word. No punctuation. No explanation.\n\n"
        f"Premise: \"{premise}\"\n"
        f"Hypothesis: \"{hypothesis}\"\n\n"
        "Label:"
    )

def call_llm(client, prompt):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config={"temperature": 0, "max_output_tokens": 10}
            )
            raw = response.text.strip().lower() if response.text else ""
            matches = re.findall(r"\b(entailment|contradiction|neutral)\b", raw)
            return matches[0] if matches else "unknown"
        except Exception as e:
            time.sleep(min(2 ** attempt, 60))
    return "error"

def run_llm_all(client, samples):
    """Run LLM on every sample for fair threshold comparison."""
    print(f"Running LLM (Zero-Shot) on all {len(samples)} samples...")
    preds = []
    for s in tqdm(samples, desc="LLM"):
        prompt = build_zero_shot(s["sentence1"], s["sentence2"])
        preds.append(call_llm(client, prompt))
        time.sleep(REQUEST_DELAY)
    return np.array(preds)

# ============================================================
# Threshold Sweep
# ============================================================
def threshold_sweep(y_test, ce_preds, ce_confs, llm_preds):
    """Simulate hybrid at different thresholds using full predictions."""
    rows = []
    for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]:
        high = ce_confs >= t
        hybrid = np.where(high, ce_preds, llm_preds)

        ce_n = high.sum()
        llm_n = len(y_test) - ce_n
        hybrid_acc = accuracy_score(y_test, hybrid)
        ce_acc = accuracy_score(y_test[high], ce_preds[high]) if ce_n > 0 else 0
        llm_acc = accuracy_score(y_test[~high], llm_preds[~high]) if llm_n > 0 else 0

        rows.append({
            "Threshold": t,
            "CE Samples": ce_n,
            "CE %": f"{ce_n/len(y_test)*100:.1f}%",
            "CE Accuracy": ce_acc,
            "LLM Samples": llm_n,
            "LLM %": f"{llm_n/len(y_test)*100:.1f}%",
            "LLM Accuracy": llm_acc,
            "Hybrid Accuracy": hybrid_acc,
        })

    return pd.DataFrame(rows)

# ============================================================
# Main
# ============================================================
def main():
    samples = load_data(DATA_FILE, TEST_SIZE)
    y_test = np.array([s["gold_label"] for s in samples])
    client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

    # Run both models on ALL samples
    ce_preds, ce_confs = run_cross_encoder(samples)
    llm_preds = run_llm_all(client, samples)

    # Baselines
    ce_acc = accuracy_score(y_test, ce_preds)
    llm_acc = accuracy_score(y_test, llm_preds)
    print(f"\nBaseline — CE Only: {ce_acc:.4f}")
    print(f"Baseline — LLM Only (Zero-Shot): {llm_acc:.4f}")

    # Sweep
    print(f"\n{'='*70}")
    print("THRESHOLD SELECTION ANALYSIS")
    print(f"{'='*70}")
    df = threshold_sweep(y_test, ce_preds, ce_confs, llm_preds)
    print(df.to_string(index=False, float_format="{:.4f}".format))

    # Confidence distribution
    print(f"\nCE Confidence Distribution:")
    print(f"  Mean: {ce_confs.mean():.4f}, Median: {np.median(ce_confs):.4f}")
    print(f"  Min: {ce_confs.min():.4f}, Max: {ce_confs.max():.4f}")

if __name__ == "__main__":
    main()
